# Improve DeepCAD

In this notebook the idea to improve DeepCAD by splitting up the CAD-sequences in smaller units will be explored.

__IMPORTANT__

- For this I disabled the sampling of point clouds to 2048 points, here I use the whole 8096 points

In [9]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply

In [10]:
def get_data(dataset, index):
    data = dataset[index]
    point_cloud = data['pc']
    sequence = data['tgt_vec']
    return point_cloud, sequence

In [11]:
def copy_to_temp(dataset, idx):
    cad_seq_path = dataset.get_cad_seq_path(idx)
    pc_path = dataset.get_pc_path(idx)
    print(idx)
    print(pc_path)
    json_path = pc_path.replace("pc_cad", "cad_json")
    json_path = json_path.replace("ply", "json")
    print(pc_path, json_path)
    
    temp_dir = "../data/temporary"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    dest_path_cad_seq = os.path.join(temp_dir, os.path.basename(cad_seq_path))
    dest_path_pc = os.path.join(temp_dir, os.path.basename(pc_path))
    dest_path_json = os.path.join(temp_dir, os.path.basename(json_path))

    shutil.copy(cad_seq_path, dest_path_cad_seq)
    shutil.copy(pc_path, dest_path_pc)
    shutil.copy(json_path, dest_path_json)
    
    print(f"Copied {cad_seq_path} to {dest_path_cad_seq}")
    print(f"Copied {pc_path} to {dest_path_pc}")
    print(f"Copied {json_path} to {dest_path_json}")
    return dest_path_json

def change_keys(h5_file):
    """Changes the keys from 'vec' to 'out_vec' in order to be able to show the sample using show.py"""
    with h5py.File(h5_file, 'r+') as hf:

        if 'vec' in hf:
            data = hf['vec'][:]
            hf.create_dataset('out_vec', data=data)
            del hf['vec']
            print(f"Changed keys from 'vec' to 'out_vec' in {h5_file}")

def export2step(json_path):
    filter = True
    save_path = os.path.join(*json_path.split("/")[:-1], os.path.splitext(os.path.basename(json_path))[0] + '.step')

    with open(json_path, "r") as fp:
        data = json.load(fp)

    cad_seq = CADSequence.from_dict(data)
    cad_seq.normalize()
    shape = create_CAD(cad_seq)

    write_step_file(shape, save_path)
    return save_path

def step2stl(step_path):

    save_path = os.path.join(*step_path.split("/")[:-1], os.path.splitext(os.path.basename(step_path))[0] + '.stl')
    step_reader = STEPControl_Reader()
    step_reader.ReadFile(step_path)
    step_reader.TransferRoots()
    shape = step_reader.OneShape()

    BRepMesh_IncrementalMesh(shape, 0.1)

    stl_writer = StlAPI_Writer()
    stl_writer.Write(shape, save_path)
    print(f"Wrote stl file to {save_path}")

def visualize_gt(dataset, idx):
    dest_path_h5 = copy_to_temp(dataset, idx)
   # change_keys(dest_path_h5)
    step_path = export2step(dest_path_h5)
    step2stl(step_path)

In [12]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [13]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0],
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0]
]
eos_row = [3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]

In [14]:
seq_np = np.array(seq, dtype=np.float32)
num_pad_rows = 60 - seq_np.shape[0]
pad_array = np.tile(eos_row, (num_pad_rows, 1))
seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
seq_len = seq_np_pad[:,0].tolist().index(3)
cad_seq = CADSequence.from_vector(seq_np_pad, is_numerical=True)
custom_shape = create_CAD(cad_seq)
write_step_file(custom_shape, "a.step")
step2stl("a.step")

Wrote stl file to a.stl

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : a.step(655 ents)  Write  Done


In [15]:
out_pc = CADsolid2pc(custom_shape, 8096, "aha")
write_ply(out_pc, "aha.ply")

In [16]:
seq_np_pad.shape

(60, 17)

In [17]:
visualize_gt(dataset, i)

NameError: name 'i' is not defined

## START

We will use the below sequence to create a minimum working example.

In [20]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

#### 1. Split CAD-sequence into extrusions

In [21]:
def seq2shape(seq):
    cad_seq = CADSequence.from_vector(seq, is_numerical=True)
    shape = create_CAD(cad_seq)
    return shape

In [22]:
def shape2cad(shape, name=None):
    if name is None:
        name = "test"
    write_step_file(shape, os.path.join("examples", name + ".step"))
    step2stl(os.path.join("examples",name + ".step"))

In [23]:
def seq2CAD(seq, name=None):
    """Takes (60,17) sequence and turns it to stl file."""
    shape = seq2shape(seq)
    shape2cad(shape, name=name)

In [24]:
def pad_seq(seq):
    """Takes custom sequence (N,17) and pads it to (60,17)"""
    eos_row = [3] + 16 * [-1]
    seq_np = np.array(seq, dtype=np.float32)
    num_pad_rows = 60 - seq_np.shape[0]
    pad_array = np.tile(eos_row, (num_pad_rows, 1))
    seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
    return seq_np_pad
    

In [25]:
def split_and_pad_sequence_by_extrusion(matrix, delimiter=5):
    """Takes (60,17) sequence and splits it by the extrusions and pads it and returns a (60,17) for each extrusion""" 
    matrix = np.array(matrix)
    assert matrix.shape == (60, 17), "Input must be (60, 17)"

    commands = matrix[:,0]
    split_indices = []
    start_idx = 0

    # Find split points
    for idx, val in enumerate(commands):
        if val == delimiter:
            split_indices.append((start_idx, idx))
            start_idx = idx + 1

    # Split and pad
    output = []
    for start, end in split_indices:
        length = 59 - (end - start)
        pad_row = [[3] + 16 * [-1]] * length
        new_matrix = matrix[start:end+1]
       
        pad_matrix = np.vstack([new_matrix, pad_row])
        output.append(pad_matrix)
    return output

In [26]:
def seq2pc(seq, nr_points=8096, name=None):
    shape = seq2shape(seq)
    if name is None:
        name = "test"
    out_pc = CADsolid2pc(shape, nr_points, name)
    write_ply(out_pc, os.path.join("examples",name + ".ply"))

In [33]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 64, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [74]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 64, 64, 192, 128, 3, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [75]:
sequence = pad_seq(seq)

In [79]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)


In [194]:
index = 31
data = dataset[index]
sequence = data["tgt_vec"].numpy()
print(sequence[:,0])

[4 0 0 0 0 5 4 1 0 5 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]


In [195]:
shape = seq2CAD(sequence, "test")


*******************************************************************
******        Statistics on Transfer (Write)                 ******
Wrote stl file to examples/test.stl

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/test.step(519 ents)  Write  Done


In [196]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)

In [197]:
for i, ext_seq in enumerate(extrusion_splits_seq):
    seq2CAD(ext_seq, name=str(i) + "_test")
    seq2pc(ext_seq, name=str(i) + "_test")
    

Wrote stl file to examples/0_test.stl
*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/0_test.step(380 ents)  Write  Done

Wrote stl file to examples/1_test.stl

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/1_test.step(226 ents)  Write  Done


## Examples

Here I keep examples

In [ ]:
# Cube with cylinder on top 
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]